In [1]:
from pathlib import Path

import pandas as pd
import seaborn as sns
from itertools import groupby

In [2]:
centromere_endpoints_mb: dict[int, tuple[int, int]] = {
    1: (14, 16),
    2: (3, 5),
    3: (13, 15),
    4: (3, 5),
    5: (11, 13),
}

In [3]:
centeromere_bins = {
    key: list(range(start * 2 - 1, end * 2 + 1 + 1))
    for key, (start, end) in centromere_endpoints_mb.items()
}
centeromere_bins

{1: [27, 28, 29, 30, 31, 32, 33],
 2: [5, 6, 7, 8, 9, 10, 11],
 3: [25, 26, 27, 28, 29, 30, 31],
 4: [5, 6, 7, 8, 9, 10, 11],
 5: [21, 22, 23, 24, 25, 26, 27]}

In [4]:
metadata_cols = [
    'Sample',
    'Type',
    'Aneuploid',
    'Chromosoes',
]

cols = [
    *metadata_cols,
    *(
        f"{chr}_{bin}"
        for chr, bins in centeromere_bins.items()
        for bin in bins
    )
]
cols

['Sample',
 'Type',
 'Aneuploid',
 'Chromosoes',
 '1_27',
 '1_28',
 '1_29',
 '1_30',
 '1_31',
 '1_32',
 '1_33',
 '2_5',
 '2_6',
 '2_7',
 '2_8',
 '2_9',
 '2_10',
 '2_11',
 '3_25',
 '3_26',
 '3_27',
 '3_28',
 '3_29',
 '3_30',
 '3_31',
 '4_5',
 '4_6',
 '4_7',
 '4_8',
 '4_9',
 '4_10',
 '4_11',
 '5_21',
 '5_22',
 '5_23',
 '5_24',
 '5_25',
 '5_26',
 '5_27']

In [5]:
def read_centromere_region_mapping(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        csv_path,
        header=0,
        index_col=0,
        usecols=lambda col: col in cols,
        skiprows=[1, 2],
    )

    df.rename(columns={'Chromosoes': 'Chromosomes'}, inplace=True)

    for meta_col in ['Type', 'Aneuploid', 'Chromosomes']:
        assert df[meta_col].nunique() == 1, f"Column {meta_col} should have a single unique value."

    return df


In [6]:
def infer_genotype(df: pd.DataFrame) -> pd.DataFrame:
    df_g = pd.DataFrame(index=df.index)

    col_groups = groupby(
        sorted(col for col in df.columns if '_' in col),
        key=lambda col: col.split('_')[0]
    )

    def majority_vote(row: pd.Series) -> str:
        """Determine the majority genotype for a given row."""

        assert len(row) >= 4
        counts = row.value_counts()
        majority = counts.index[0]
        assert isinstance(majority, str)

        if counts[majority] >= len(row) - 1:
            return majority

        return "Unknown"

    for chr, cols in col_groups:
        df_g[chr] = df[list(cols)].apply(majority_vote, axis=1)

    return df_g

In [7]:
df_aa_c = read_centromere_region_mapping(Path("../hamburg-genetic-mapping/AA_C.csv"))

df_aa_c

,Type,Aneuploid,Chromosomes,1_27,1_28,1_29,1_31,1_32,1_33,2_5,...,4_9,4_10,4_11,5_21,5_22,5_23,5_24,5_25,5_26,5_27
Sample,,,,,,,,,,,,,,,,,,,,,
aa-C-101,aa-C,No,2x,A,A,A,A,A,A,H,...,A,A,A,A,A,A,A,A,A,A
aa-C-103,aa-C,No,2x,A,A,A,NaN,A,A,A,...,A,A,A,A,A,A,A,A,A,A
aa-C-105,aa-C,No,2x,H,H,H,NaN,H,H,A,...,H,H,H,H,H,H,H,H,H,H
aa-C-106,aa-C,No,2x,H,H,H,H,H,H,H,...,A,A,A,A,A,A,A,A,A,A
aa-C-107,aa-C,No,2x,H,H,H,H,H,H,H,...,H,H,H,A,A,A,A,A,A,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
aa-C-95,aa-C,No,2x,H,H,H,H,H,H,H,...,A,A,A,A,A,A,A,A,A,A
aa-C-96,aa-C,No,2x,H,H,H,A,H,H,H,...,A,A,A,A,A,A,A,A,A,A
aa-C-97,aa-C,No,2x,A,A,A,A,A,A,H,...,A,A,A,A,A,A,A,A,A,A


In [8]:
infer_genotype(df_aa_c)

,1,2,3,4,5
Sample,,,,,
aa-C-101,A,H,H,A,A
aa-C-103,A,A,A,A,A
aa-C-105,H,A,A,H,H
aa-C-106,H,H,A,A,A
aa-C-107,H,H,A,H,A
...,...,...,...,...,...
aa-C-95,H,H,A,A,A
aa-C-96,H,H,A,A,A
aa-C-97,A,H,A,A,A


In [9]:
out_dir = Path("../data/hamburg/cen-genotypes")
out_dir.mkdir(parents=True, exist_ok=True)

for csv_path in Path("../hamburg-genetic-mapping").glob("*.csv"):
    df = read_centromere_region_mapping(csv_path)
    df_g = infer_genotype(df)
    df_g.to_csv(out_dir / (csv_path.stem + '.cen-genotype.csv'), index=True)